# AQIS lab: Sprint 3 cascade routing and LoRA pilot (Colab)

Runs one stage of WP-1.3 (`docs/specs/sprints/WP-1.3-S3-cascade-finetune.md`) as the lab commands do on any machine: mount Drive, clone the private repository at the tag `AQIS_TAG`, install the locked environments, run `python -m aqis_lab.cascade <action>`, write the unit log, and free the runtime.

**Before you start:**
- Sprint 1 must be complete on Drive: `MyDrive/AQIS_LAB/s1/data/` and `.../s1/preds/` (the cascade simulation reads them, and `revisions.json` pins the model revisions).
- Colab Secrets: `HF_TOKEN` and `GITHUB_TOKEN`, as in Sprints 1 and 2. Drive: about 8 GB free for the training set and adapter.
- Budget: the brief allows about 20 to 30 units for the whole sprint. `cascade`, `prepare` and `report` run on a CPU runtime; `confidence`, `prompt`, `train` and `eval` need a GPU.

**Each session:** edit only cell 1, pick the runtime below, then **Run all**. For `confidence`, `prompt`, `train` and `eval`, run once with `SMOKE = True` first (3 turns, or 4 training steps, into `AQIS_LAB/s3/smoke`; it never touches the real files), read the output, then set it back to `False`. The last cell writes the unit log and, when every action exited 0, frees the runtime.

| STAGE | Runtime | What it does |
| --- | --- | --- |
| `cascade` | CPU | simulates the gate policies over the Sprint 1 predictions (with `NONSPEECH = True` it first computes the Silero v5 non-speech ratio of the 6,300 target turns, a few minutes) and writes `report/cascade.csv`, `headline.csv`, `thresholds.json` |
| `confidence` | **L4 GPU** | saves parakeet-tdt-0.6b-v2 token confidences for the target turns (the one gate feature the Sprint 1 files lack); about 15 minutes; run `cascade` again afterwards |
| `prepare` | CPU | writes the biasing terms (drug names from Eka speakers outside the evaluated slice) and builds the speaker-disjoint Eka training and validation sets (augmented to clean, phone and noise_10) with their manifest |
| `prompt` | **L4 GPU** | keyword and context biasing of granite and qwen3-asr on `medical` and `numbers` (clean, phone, noise_10); about 30 minutes; needs `prepare` first |
| `train` | **L4 or A100 GPU** | QLoRA (r 16, alpha 32) on Qwen3-ASR 1.7B with early stopping; resumable after a dropped session; the brief expects 2 to 3 hours |
| `eval` | **L4 GPU** | base against merged adapter on medical, us_conv and numbers (clean, phone); about 20 minutes |
| `report` | CPU | scorecard, `tradeoff.png`, `prompt.csv`, `finetune.csv`, `decision_inputs.md` and `report/s3_report.zip` |

Order: `cascade` (no confidence yet), `confidence`, `cascade` again, `prepare`, `prompt`, `train`, `eval`, `report`. Use an L4; ask before an A100.

In [ ]:
# 1) Settings: edit STAGE and the options only. Then Google Drive is mounted.
AQIS_TAG = "s3-colab-1"
STAGE = "cascade"       # cascade | confidence | prompt | prepare | train | eval | report
NONSPEECH = True        # cascade only: compute the Silero v5 non-speech ratio first (needs the s3-cascade lock; a few minutes)
PROMPT_MODELS = ["granite-speech-4.1-2b", "qwen3-asr-1.7b"]
SMOKE = False           # confidence, prompt, train, eval: True runs 3 turns (train: 4 steps) into AQIS_LAB/s3/smoke; run it before the real stage

REPO = "https://github.com/NasirAman-promcare/voice-bot-vad-harness.git"
WORKDIR = "/content/aqis"
SCRATCH = "/content/s3_work"
DRIVE_MOUNT = "/content/drive"
AQIS_DRIVE = "/content/drive/MyDrive/AQIS_LAB"
S1_OUT = AQIS_DRIVE + "/s1"
S3_OUT = AQIS_DRIVE + "/s3"

STAGES = ("cascade", "confidence", "prompt", "prepare", "train", "eval", "report")
assert STAGE in STAGES, f"STAGE must be one of {STAGES}"

from google.colab import drive

drive.mount(DRIVE_MOUNT)

In [ ]:
# 2) Clone the private repository at AQIS_TAG. The GITHUB_TOKEN secret is read inside this one git command's
#    environment; it is never stored in a variable, a URL, .git/config or any output.
import base64
import os
import subprocess

from google.colab import userdata

subprocess.run(["rm", "-rf", WORKDIR], check=True)
subprocess.run(["git", "clone", "--quiet", "--depth", "1", "--branch", AQIS_TAG, REPO, WORKDIR],
               env=dict(os.environ, GIT_TERMINAL_PROMPT="0", GIT_CONFIG_COUNT="1", GIT_CONFIG_KEY_0="http.extraHeader",
                        GIT_CONFIG_VALUE_0="Authorization: Basic " + base64.b64encode(
                            ("x-access-token:" + userdata.get("GITHUB_TOKEN")).encode()).decode()),
               check=True)

In [ ]:
# 3) Environments. The lab's own .venv (hash-checked phase 0 lock) always runs the unit log. cascade, prepare and report also get the
#    s3-cascade lock on top (prepare reuses the Sprint 1 data code). Each GPU stage builds its own venv from a hash-pinned lock:
#    confidence uses the Sprint 1 NeMo lock, prompt one venv per model stack (Sprint 1 locks), train and eval the s3-train lock.
#    A failed install is printed and recorded in SETUP_ERROR, so the last cell still frees the runtime.
import re

import yaml

PY = WORKDIR + "/.venv/bin/python"
LOCKS = WORKDIR + "/requirements/aqis_lab"
CPU_STAGES = ("cascade", "prepare", "report")
STACK_PY = {}
SETUP_ERROR = None


def make_venv(name, lock, python_version):
    venv = f"/content/venv-{name}"
    subprocess.run(["uv", "venv", "--clear", "--python", python_version, venv], check=True)
    subprocess.run(["uv", "pip", "sync", "--python", venv + "/bin/python", "--require-hashes", "--build-constraints",
                    LOCKS + "/phase0.txt", f"{LOCKS}/{lock}"], check=True)
    print("ready:", name, venv)
    return venv + "/bin/python"


try:
    subprocess.run(["pip", "install", "--quiet", "uv==0.8.17"], check=True)
    subprocess.run(["bash", "aqis_lab/deploy/dev_bootstrap.sh"], cwd=WORKDIR, check=True)
    s1 = yaml.safe_load(open(WORKDIR + "/config/aqis_lab/s1.yaml"))
    if STAGE in CPU_STAGES:
        subprocess.run(["uv", "pip", "install", "--python", PY, "--require-hashes", "--build-constraints", LOCKS + "/phase0.txt",
                        "-r", LOCKS + "/s3-cascade.txt"], check=True)
    else:
        print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                             capture_output=True, text=True, check=True).stdout.strip())
        probe = open(WORKDIR + "/docs/environment/colab_probe.txt").read()
        colab_python = re.search(r"Python Version:\s*(\d+\.\d+)", probe).group(1)
        if STAGE == "confidence":
            STACK_PY["nemo"] = make_venv("nemo", "s1-nemo.txt", colab_python)
        elif STAGE == "prompt":
            unknown = [m for m in PROMPT_MODELS if m not in s1["models"]]
            assert not unknown, f"not in s1.yaml: {unknown}"
            for stack in sorted({s1["models"][m]["stack"] for m in PROMPT_MODELS}):
                STACK_PY[stack] = make_venv(stack, f"s1-{stack}.txt", colab_python)
        else:
            STACK_PY["train"] = make_venv("train", "s3-train.txt", colab_python)
except (subprocess.CalledProcessError, AssertionError) as exc:
    SETUP_ERROR = repr(exc)
    print("SETUP FAILED:", SETUP_ERROR)

In [ ]:
# 4) Unit log start (the runtime is billed from here), then the stage. HF_TOKEN goes into each subprocess's environment
#    only. The output streams line by line. The last cell frees the runtime only if every action exits 0.
env = dict(os.environ, HF_TOKEN=userdata.get("HF_TOKEN"), PYTHONUNBUFFERED="1", PYTHONPATH=WORKDIR, MPLBACKEND="Agg")
results = {}
smoke = ["--smoke"] if SMOKE else []


def cascade(python, action, *extra):
    """Run one aqis_lab.cascade action and stream its output; returns the exit code."""
    process = subprocess.Popen([python, "-u", "-m", "aqis_lab.cascade", action, "--out", S3_OUT, "--s1-out", S1_OUT,
                                "--work-dir", SCRATCH, *extra], cwd=WORKDIR, env=env, stdout=subprocess.PIPE,
                               stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in process.stdout:
        print(line, end="", flush=True)
    return process.wait()


if SETUP_ERROR:
    print("Skipping the stage because setup failed:", SETUP_ERROR)
else:
    subprocess.run([PY, "-m", "aqis_lab", "unitlog", "start", "--drive-mount", DRIVE_MOUNT], cwd=WORKDIR, check=True)
    if STAGE == "cascade":
        results[STAGE] = cascade(PY, "cascade", *(["--nonspeech"] if NONSPEECH else []))
    elif STAGE == "prepare":
        results[STAGE] = cascade(PY, "train", "--prepare-only")
    elif STAGE == "report":
        results[STAGE] = cascade(PY, "report")
    elif STAGE == "confidence":
        results[STAGE] = cascade(STACK_PY["nemo"], "confidence", *smoke)
    elif STAGE == "prompt":
        for model in PROMPT_MODELS:
            print("=" * 20, "prompt", model)
            results[model] = cascade(STACK_PY[s1["models"][model]["stack"]], "prompt", "--models", model, *smoke)
    elif STAGE == "train":
        results[STAGE] = cascade(STACK_PY["train"], "train", "--skip-prepare", *smoke)
    else:
        results[STAGE] = cascade(STACK_PY["train"], "eval", *smoke)
    print("exit codes (0 = done; 2 = error):")
    for name, rc in results.items():
        print(f"  {name:28s} {rc}")

In [ ]:
# 5) End of session: append this session's estimated compute units to AQIS_LAB/logs/units.csv. The runtime is freed
#    (so the GPU stops costing units) only when setup worked and every action above exited 0; after any failure it stays
#    connected so you can read the output and debug. Then free it yourself: Runtime > Disconnect and delete runtime.
try:
    subprocess.run([PY, "-m", "aqis_lab", "unitlog", "end", "--drive-mount", DRIVE_MOUNT], cwd=WORKDIR, check=True)
except subprocess.CalledProcessError as exc:
    print("UNIT LOG NOT WRITTEN, note this session's hours by hand:", exc)
if SETUP_ERROR is None and results and all(rc == 0 for rc in results.values()):
    from google.colab import runtime

    runtime.unassign()
else:
    print("NOT freeing the runtime: setup failed or an action exited non-zero (see the exit codes above). "
          "A GPU runtime keeps costing units until you disconnect it.")